# Time Series in Pandas

Time series data contains observations indexed by time.
Pandas provides powerful tools for analyzing date-based data.

In [2]:
import pandas as pd, numpy as np

In [3]:
df = pd.read_csv("time_series_data.csv")
df.head(10)

,Date,Product_Category,Units_Sold,Unit_Price,Revenue
0,2026-01-01,Electronics,12,150,1800
1,2026-01-02,Electronics,15,150,2250
2,2026-01-03,Electronics,22,150,3300
3,2026-01-04,Electronics,28,150,4200
4,2026-01-05,Electronics,10,150,1500
5,2026-01-06,Electronics,9,150,1350
6,2026-01-07,Electronics,14,150,2100
7,2026-01-08,Electronics,18,150,2700
8,2026-01-09,Electronics,20,150,3000
9,2026-01-10,Electronics,35,150,5250


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28 entries, 0 to 27
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Date              28 non-null     object
 1   Product_Category  28 non-null     object
 2   Units_Sold        28 non-null     int64 
 3   Unit_Price        28 non-null     int64 
 4   Revenue           28 non-null     int64 
dtypes: int64(3), object(2)
memory usage: 1.2+ KB


## Convert to Datetime
Datetime conversion enables powerful time-based indexing,
resampling, and date component extraction.

In [5]:
df['Date'] = pd.to_datetime(df['Date'])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28 entries, 0 to 27
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Date              28 non-null     datetime64[ns]
 1   Product_Category  28 non-null     object        
 2   Units_Sold        28 non-null     int64         
 3   Unit_Price        28 non-null     int64         
 4   Revenue           28 non-null     int64         
dtypes: datetime64[ns](1), int64(3), object(1)
memory usage: 1.2+ KB


## Set Date as Index

In [6]:
df.set_index('Date', inplace=True)
df.head()

,Product_Category,Units_Sold,Unit_Price,Revenue
Date,,,,
2026-01-01,Electronics,12,150,1800
2026-01-02,Electronics,15,150,2250
2026-01-03,Electronics,22,150,3300
2026-01-04,Electronics,28,150,4200
2026-01-05,Electronics,10,150,1500


## Sort the index to ensure it is monotonic

In [7]:
df = df.sort_index()

## Filtering by Date

In [8]:
df.loc['2026-01-05']

Product_Category    Electronics
Units_Sold                   10
Unit_Price                  150
Revenue                    1500
Name: 2026-01-05 00:00:00, dtype: object

In [9]:
df.loc['2026-01-02':'2026-01-08']

,Product_Category,Units_Sold,Unit_Price,Revenue
Date,,,,
2026-01-02,Electronics,15,150,2250
2026-01-03,Electronics,22,150,3300
2026-01-04,Electronics,28,150,4200
2026-01-05,Electronics,10,150,1500
2026-01-06,Electronics,9,150,1350
2026-01-07,Electronics,14,150,2100
2026-01-08,Electronics,18,150,2700


## Extract Date Components

### Year

In [10]:
df['year'] = df.index.year

### Month

In [11]:
df['month'] = df.index.month

### Day

In [12]:
df['day'] = df.index.day

## Resampling
`resample()` groups data based on time frequency,
while `groupby()` groups based on categorical values.

### Downsampling (Going "Up" in Time)
Aggregate daily data into weekly totals

Common resampling frequencies:
- 'D' → Daily
- 'M' → Monthly
- 'Y' → Yearly

In [13]:
df['Revenue'].resample('W').sum()

Date
2026-01-04    11550
2026-01-11    21900
2026-01-18    24300
2026-01-25    27000
2026-02-01     7950
Freq: W-SUN, Name: Revenue, dtype: int64

#### Resample by week and get multiple stats
- Sum: Total sales for the week.
- Mean: Average daily sale that week.
- Std: How much sales fluctuated (volatility) during that week.

In [14]:
weekly_summary = df['Revenue'].resample('W').agg(['sum', 'mean', 'std'])
weekly_summary

,sum,mean,std
Date,,,
2026-01-04,11550,2887.500000,1077.323071
2026-01-11,21900,3128.571429,1817.474700
2026-01-18,24300,3471.428571,1988.897756
2026-01-25,27000,3857.142857,2192.112484
2026-02-01,7950,2650.000000,377.491722


### Upsampling (Going "Down" in Time)
Stretch daily data to hourly (creates 24 rows for every 1 day)

In [15]:
df['Revenue'].resample('H').ffill()

C:\Users\hitek\AppData\Local\Temp\ipykernel_8964\201211952.py:1: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df['Revenue'].resample('H').ffill()


Date
2026-01-01 00:00:00    1800
2026-01-01 01:00:00    1800
2026-01-01 02:00:00    1800
2026-01-01 03:00:00    1800
2026-01-01 04:00:00    1800
                       ... 
2026-01-27 20:00:00    2250
2026-01-27 21:00:00    2250
2026-01-27 22:00:00    2250
2026-01-27 23:00:00    2250
2026-01-28 00:00:00    3000
Freq: h, Name: Revenue, Length: 649, dtype: int64

## Basic Rolling Mean
Calculate 7-day Moving Average

In [16]:
df['Rolling_Avg'] = df['Revenue'].rolling(window=7).mean()
df.head(10)

,Product_Category,Units_Sold,Unit_Price,Revenue,year,month,day,Rolling_Avg
Date,,,,,,,,
2026-01-01,Electronics,12,150,1800,2026,1,1,NaN
2026-01-02,Electronics,15,150,2250,2026,1,2,NaN
2026-01-03,Electronics,22,150,3300,2026,1,3,NaN
2026-01-04,Electronics,28,150,4200,2026,1,4,NaN
2026-01-05,Electronics,10,150,1500,2026,1,5,NaN
2026-01-06,Electronics,9,150,1350,2026,1,6,NaN
2026-01-07,Electronics,14,150,2100,2026,1,7,2357.142857
2026-01-08,Electronics,18,150,2700,2026,1,8,2485.714286
2026-01-09,Electronics,20,150,3000,2026,1,9,2592.857143


### Centered Windows
By default, the "window" looks backward. If you want the average to be based on the 3 days before, the current day, and the 3 days after, use `center=True`.

In [17]:
df['7day_Centered'] = df['Revenue'].rolling(window=7, center=True).sum()
df.head(10)

,Product_Category,Units_Sold,Unit_Price,Revenue,year,month,day,Rolling_Avg,7day_Centered
Date,,,,,,,,,
2026-01-01,Electronics,12,150,1800,2026,1,1,NaN,NaN
2026-01-02,Electronics,15,150,2250,2026,1,2,NaN,NaN
2026-01-03,Electronics,22,150,3300,2026,1,3,NaN,NaN
2026-01-04,Electronics,28,150,4200,2026,1,4,NaN,16500.0
2026-01-05,Electronics,10,150,1500,2026,1,5,NaN,17400.0
2026-01-06,Electronics,9,150,1350,2026,1,6,NaN,18150.0
2026-01-07,Electronics,14,150,2100,2026,1,7,2357.142857,20100.0
2026-01-08,Electronics,18,150,2700,2026,1,8,2485.714286,21900.0
2026-01-09,Electronics,20,150,3000,2026,1,9,2592.857143,22200.0


### Handling NaN (min_periods)
If you don't want those first few rows to be empty, use `min_periods`.

In [18]:
df['7day_Robust'] = df['Revenue'].rolling(window=7, min_periods=1).mean()
df.head(10)

,Product_Category,Units_Sold,Unit_Price,Revenue,year,month,day,Rolling_Avg,7day_Centered,7day_Robust
Date,,,,,,,,,,
2026-01-01,Electronics,12,150,1800,2026,1,1,NaN,NaN,1800.000000
2026-01-02,Electronics,15,150,2250,2026,1,2,NaN,NaN,2025.000000
2026-01-03,Electronics,22,150,3300,2026,1,3,NaN,NaN,2450.000000
2026-01-04,Electronics,28,150,4200,2026,1,4,NaN,16500.0,2887.500000
2026-01-05,Electronics,10,150,1500,2026,1,5,NaN,17400.0,2610.000000
2026-01-06,Electronics,9,150,1350,2026,1,6,NaN,18150.0,2400.000000
2026-01-07,Electronics,14,150,2100,2026,1,7,2357.142857,20100.0,2357.142857
2026-01-08,Electronics,18,150,2700,2026,1,8,2485.714286,21900.0,2485.714286
2026-01-09,Electronics,20,150,3000,2026,1,9,2592.857143,22200.0,2592.857143


## Resampling vs. Rolling
| Feature | Resampling (`.resample`) | Rolling (`.rolling`) |
|---------|--------------------------|--------------------------|
| Output Size | Shrinks the data (Daily → Weekly) | Keeps the same number of rows |
| Movement | Jumps by the time interval | Slides one row at a time |
| Purpose | Aggregating for reporting | Smoothing for trend analysis |

## Summary

- **Datetime Casting**: Raw date strings are just text; converting to `datetime64` unlocks the `.dt` accessor and time-aware indexing.
- **The Power of the Index**: Setting the date as the index transforms the DataFrame from a standard table into a **Time Series object**, allowing for partial string slicing (e.g., `df.loc['2026-01']`).
- **Resampling for Reporting**: Resampling is essential for "Business Intelligence" (e.g., converting 365 daily rows into 12 monthly rows for a financial report).
- **Rolling Windows for Trends**: Real-world data is "noisy." Rolling averages are the primary tool used to strip away daily fluctuations to reveal the actual long-term trend.
- **Handling Boundary Data**: Using `min_periods` in rolling windows is a best practice to avoid losing data at the start of your dataset.